# Notebook 01 — Naive OLS and the Omitted Variable Bias Formula

**Question:** What is the return to an additional year of schooling on monthly earnings?

**The challenge:** Higher-ability individuals both pursue more schooling *and* earn higher wages. A naive regression of wages on schooling conflates the effect of education with the effect of ability — producing an upward-biased estimate.

**What makes this dataset unusual:** We have IQ scores. Most observational datasets don't include any proxy for cognitive ability, which means you can usually only *assert* the direction of OVB. Here we can *calculate* it — applying the Omitted Variable Bias formula numerically and verifying the result holds to floating-point precision.

**Notebooks in this project:**
- **01 (this):** Naive OLS → confounding structure → progressive controls → OVB formula
- **02:** Frisch-Waugh-Lovell theorem — the mechanical explanation of *how* OLS removes bias
- **03:** Nonlinearities (log wages, Mincer quadratic) and control selection trade-offs
- **04:** Fixed effects via de-meaning and regression as variance-weighted average

In [ ]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import wooldridge as woo

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. The Data

Wooldridge (1993) `wage2`: National Longitudinal Survey of Youth data on 935 young men.
Key variables: monthly wages, years of schooling, work experience, tenure, and — crucially — IQ scores.

In [ ]:
df = woo.dataWoo('wage2').copy()
df['log_wage'] = np.log(df['wage'])

print(f"Full sample: N = {len(df)}")
print(f"IQ non-missing: N = {df['IQ'].notna().sum()}")
print()
print(df[['wage', 'log_wage', 'educ', 'exper', 'IQ', 'sibs']].describe().round(2))

## 2. The Naive Estimate

Start with the simplest possible model: log wages regressed on years of schooling alone.
This is the "freshman regression" — the number everyone cites before they think about identification.

In [ ]:
naive = smf.ols('log_wage ~ educ', data=df).fit()

print("=== Naive OLS: log(wage) ~ educ ===")
print(naive.summary().tables[1])
print()
print(f"Naive interpretation: each additional year of schooling raises")
print(f"monthly wages by approximately {naive.params['educ']*100:.1f}%.")
print(f"This estimate is almost certainly too high.")

## 3. The Confounding Structure

Why is the naive estimate biased? The causal graph:

```
Ability (IQ) ──→ Schooling ──→ log(wage)
    └─────────────────────────→ log(wage)
```

Ability is a **common cause**: it drives people to pursue more schooling *and* makes them more productive workers. The backdoor path `Ability → log(wage)` is open, and any variation in wages that is actually caused by ability gets attributed to schooling in the naive regression.

We can see this directly: workers with higher IQ scores have more schooling *and* higher wages, even before accounting for schooling's own effect.

In [ ]:
df_iq = df.dropna(subset=['IQ']).copy()
df_iq['IQ_quartile'] = pd.qcut(df_iq['IQ'], q=4, labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)'])

# Verify confounding: IQ predicts both treatment and outcome
tab = df_iq.groupby('IQ_quartile')[['educ', 'log_wage']].mean().round(3)
tab.columns = ['Mean Years Schooling', 'Mean Log Wage']
print("IQ quartile averages — both education and wages rise with IQ:")
print(tab)
print()
print("This is the classic confounding pattern:")
print("high-IQ workers get more schooling AND earn more, opening a backdoor path.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = sns.color_palette('coolwarm', 4)
iq_labels = ['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']

for i, label in enumerate(iq_labels):
    group = df_iq[df_iq['IQ_quartile'] == label]
    axes[0].scatter(group['educ'], group['log_wage'], alpha=0.35,
                    color=palette[i], s=18, label=label)

x_range = np.linspace(df['educ'].min(), df['educ'].max(), 200)
axes[0].plot(x_range,
             naive.params['Intercept'] + naive.params['educ'] * x_range,
             color='black', linewidth=2.5, label='Naive OLS')
axes[0].set_xlabel('Years of Schooling', fontsize=11)
axes[0].set_ylabel('Log Monthly Wage', fontsize=11)
axes[0].set_title('Naive Regression\n(IQ-colored: higher IQ clusters upper-right)', fontsize=11)
axes[0].legend(title='IQ Quartile', fontsize=8, title_fontsize=8)

# Right panel: average log wage by (educ, IQ quartile)
pivot = df_iq.groupby(['educ', 'IQ_quartile'])['log_wage'].mean().reset_index()
for i, label in enumerate(iq_labels):
    sub = pivot[pivot['IQ_quartile'] == label]
    axes[1].plot(sub['educ'], sub['log_wage'], color=palette[i], marker='o',
                 markersize=5, linewidth=1.5, label=label, alpha=0.85)
axes[1].set_xlabel('Years of Schooling', fontsize=11)
axes[1].set_ylabel('Mean Log Monthly Wage', fontsize=11)
axes[1].set_title('Mean Wage by Schooling and IQ Quartile\n(ability shifts the whole wage profile upward)', fontsize=11)
axes[1].legend(title='IQ Quartile', fontsize=8, title_fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/01_confounding_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Progressive Control Addition

Each control set we add blocks one or more backdoor paths. Watch the schooling coefficient fall as confounding is absorbed.

In [ ]:
# Work on the IQ-nonmissing subsample throughout so sample doesn't change across specs
iq_sample = df.dropna(subset=['IQ']).copy()

specs = {
    'Naive': 'log_wage ~ educ',
    '+ Experience (linear)': 'log_wage ~ educ + exper',
    '+ Experience (quadratic)': 'log_wage ~ educ + exper + I(exper**2)',
    '+ Tenure & Demographics': 'log_wage ~ educ + exper + I(exper**2) + tenure + married + black + south + urban',
    '+ IQ (ability proxy)': 'log_wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban',
}

fitted = {name: smf.ols(formula, data=iq_sample).fit() for name, formula in specs.items()}

results = pd.DataFrame({
    name: {
        'β_educ': m.params['educ'],
        'SE': m.bse['educ'],
        '95% CI lower': m.conf_int().loc['educ', 0],
        '95% CI upper': m.conf_int().loc['educ', 1],
        'R²': m.rsquared,
        'N': int(m.nobs),
    }
    for name, m in fitted.items()
}).T.round(4)

print("Return to schooling across control specifications (same N throughout):\n")
print(results.to_string())

In [ ]:
# Coefficient plot: β_educ with 95% CIs across specifications
fig, ax = plt.subplots(figsize=(9, 4.5))
names = list(fitted.keys())
betas = [fitted[n].params['educ'] for n in names]
lo = [fitted[n].conf_int().loc['educ', 0] for n in names]
hi = [fitted[n].conf_int().loc['educ', 1] for n in names]

colors = ['#d62728' if n == 'Naive' else ('#1f77b4' if '+ IQ' in n else '#7f7f7f') for n in names]
y_pos = list(range(len(names)))

for i, (b, l, h, c) in enumerate(zip(betas, lo, hi, colors)):
    ax.plot([l, h], [i, i], color=c, linewidth=2.5, alpha=0.8)
    ax.scatter([b], [i], color=c, s=80, zorder=5)

ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Coefficient on Years of Schooling (log wage scale)', fontsize=11)
ax.set_title('Return to Schooling Falls as Controls Absorb Confounding Bias', fontsize=12)
ax.invert_yaxis()

ax.annotate('Naive: ~6%\nper year', xy=(betas[0], 0), xytext=(betas[0]+0.003, 0.35),
            fontsize=9, color='#d62728')
ax.annotate('With IQ: ~4.5%\nper year', xy=(betas[-1], len(names)-1),
            xytext=(betas[-1]+0.003, len(names)-1.5), fontsize=9, color='#1f77b4')

plt.tight_layout()
plt.savefig('../outputs/01_coefficient_progression.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. The Omitted Variable Bias Formula

The drop in the schooling coefficient when we add IQ is not mysterious — it has a precise algebraic explanation.

**Setup:** Suppose the true (long) model is:

$$\log(\text{wage}) = \tau \cdot \text{educ} + \delta \cdot \text{IQ} + \theta X + \varepsilon$$

But we estimate the short model omitting IQ:

$$\log(\text{wage}) = \alpha \cdot \text{educ} + \theta X + u$$

**The OVB theorem** (Frisch & Waugh 1933) states:

$$\alpha = \tau + \underbrace{\delta \cdot \gamma}_{\text{omitted variable bias}}$$

where $\gamma$ is the coefficient on `educ` in the auxiliary regression `IQ ~ educ + X`.

**Interpretation of each piece:**
- $\tau$: the *true* causal return to schooling (holding IQ fixed)
- $\delta$: how much IQ raises wages (the "damage" of omitting it)
- $\gamma$: how much schooling predicts IQ after partialling out X (the "exposure" — how correlated is the omitted variable with the treatment?)
- $\alpha$: what OLS actually estimates when IQ is omitted — $\tau$ plus the compound bias

In [ ]:
controls = 'exper + I(exper**2) + tenure + married + black + south + urban'

# The three models required by the OVB formula
short_model  = smf.ols(f'log_wage ~ educ + {controls}',          data=iq_sample).fit()  # α
long_model   = smf.ols(f'log_wage ~ educ + IQ + {controls}',     data=iq_sample).fit()  # τ, δ
aux_model    = smf.ols(f'IQ ~ educ + {controls}',                data=iq_sample).fit()  # γ

tau   = long_model.params['educ']   # true return to schooling (conditional on IQ)
delta = long_model.params['IQ']     # IQ's effect on wages
gamma = aux_model.params['educ']    # how much educ predicts IQ, partialling out X

bias          = delta * gamma
predicted_alpha = tau + bias
actual_alpha  = short_model.params['educ']

print("=== Omitted Variable Bias Formula ===\n")
print(f"Long model β_educ  (τ):           {tau:.6f}   ← true return to schooling")
print(f"IQ coefficient     (δ):           {delta:.6f}   ← wage premium per IQ point")
print(f"educ→IQ coefficient (γ):          {gamma:.6f}   ← IQ points per year of schooling")
print()
print(f"Bias = δ × γ:                     {bias:.6f}")
print(f"OVB formula prediction (τ + δγ):  {predicted_alpha:.6f}")
print(f"Actual short-model α:             {actual_alpha:.6f}")
print()
print(f"Match (to 1e-10): {np.isclose(predicted_alpha, actual_alpha, atol=1e-10)}")

In [ ]:
# Visualize the decomposition
fig, ax = plt.subplots(figsize=(8, 4))

labels = ['True return\n(τ)', 'Ability bias\n(δ × γ)', 'Naive estimate\n(α = τ + δγ)']
values = [tau, bias, actual_alpha]
bar_colors = ['#1f77b4', '#d62728', '#7f7f7f']

bars = ax.bar(labels, values, color=bar_colors, width=0.5, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Coefficient on Years of Schooling', fontsize=11)
ax.set_title('OVB Decomposition: Naive Estimate = True Effect + Ability Bias', fontsize=12)
ax.set_ylim(0, max(values) * 1.25)

# Arrow showing decomposition
ax.annotate('', xy=(1, tau + bias/2), xytext=(0, tau),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            annotation_clip=False)
ax.text(0.5, tau + bias/2 + 0.001, '+', fontsize=14, ha='center', color='gray')

plt.tight_layout()
plt.savefig('../outputs/01_ovb_decomposition.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("Interpretation:")
print(f"  The naive estimate of {actual_alpha*100:.1f}% per year of schooling")
print(f"  overstates the true effect of {tau*100:.1f}% by {bias*100:.1f} percentage points.")
print(f"  This bias arises because each year of schooling is associated with")
print(f"  {gamma:.1f} additional IQ points (γ), and each IQ point raises wages")
print(f"  by {delta*100:.2f}% (δ). The compound effect: {delta:.4f} × {gamma:.2f} = {bias:.4f}.")

## 6. What Have We Established?

1. **Naive OLS** produces an upward-biased estimate of the return to schooling because ability (proxied by IQ) confounds the schooling-wage relationship.

2. **The OVB formula** decomposes this bias precisely: it equals the IQ wage premium times the education-IQ correlation. The formula matches the actual bias to ten decimal places — not an approximation.

3. **Adding controls progressively** absorbs the confounding. The estimate drops from ~6% to ~4.5% per year of schooling, with IQ doing most of the work.

**What we haven't explained yet:** *Why* does adding IQ to the regression remove the confounding? The OVB formula tells us the size of the bias. The Frisch-Waugh-Lovell theorem — in the next notebook — explains the mechanical process by which OLS actually eliminates it.

## 6. Contextualizing the Estimate

We find a return of approximately **4.5% per additional year of schooling** (coefficient on `educ` in the fully controlled model). How does this compare to the literature?

The Mincer (1974) and subsequent OLS literature typically finds returns in the **7–10% range** using U.S. data. Our estimate is lower for two reasons:

1. **IQ controls out a lot.** Most OLS studies don't include any direct measure of cognitive ability, so their estimates carry more ability bias than ours. Including IQ shifts the estimate from ~6% to ~4.5% — the ~1.5pp gap is exactly the ability bias we quantified above.

2. **This is a young-men sample (NLS-Y, ages 28–38).** Returns to schooling estimated on workers early in their careers are typically lower than lifecycle estimates, partly because the wage variance explained by schooling hasn't fully materialized yet.

**The IV comparison.** Card (1995) uses proximity to a four-year college as an instrument for schooling and finds returns of **9–13%** — substantially *higher* than OLS estimates like ours. This initially seems backwards: if OLS is biased upward by ability, why does IV give a higher number?

The answer is two-part. First, IV estimates a Local Average Treatment Effect (LATE): the return for people whose schooling was affected by proximity to a college — likely credit-constrained individuals who would otherwise not have attended. Their returns may be higher than the population average (heterogeneous treatment effects). Second, OLS attenuation bias from measurement error in reported years of schooling pushes the OLS estimate *down*, partially offsetting the upward ability bias. The direction of net OLS bias is theoretically ambiguous, and our ~4.5% estimate sits below Card's IV estimate — consistent with measurement error being the dominant OLS bias in this specification.

**Takeaway for the portfolio:** The ~4.5% estimate shouldn't be treated as a definitive causal number (CIA is not fully plausible), but the OVB decomposition exercise is valid regardless: we demonstrated that a 1.5pp gap between naive and controlled OLS is predicted exactly by the ability channel, which is the pedagogical goal.